<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M09/M09_Lab2_Streamlit_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">📊 M09 Lab 2 — Streamlit AI Dashboard</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand <b>Streamlit</b> and how it differs from Gradio</li>
    <li>Build an AI-powered <b>data dashboard</b> with CSV upload</li>
    <li>Generate <b>AI insights</b> from uploaded data</li>
    <li>Create <b>interactive charts</b> based on AI recommendations</li>
    <li>Run Streamlit from Colab using a <b>tunnel workaround</b></li>
  </ol>
</div>

## 📦 Setup

> **⚠️ Important Note:** Streamlit doesn't run natively inside Colab cells like Gradio. We'll write the Streamlit app to a `.py` file and run it using a tunnel. Follow the instructions carefully.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai streamlit pandas matplotlib
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, quiz

client = setup_openai()

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Gradio vs Streamlit</h2>
</div>

Both are great for AI prototyping, but they shine in different areas:

| Feature | Gradio | Streamlit |
|---------|--------|-----------|
| **Best for** | Quick demos, single-function apps | Multi-page dashboards |
| **Colab support** | Native (runs inline) | Needs tunnel workaround |
| **State management** | Limited | `st.session_state` (powerful) |
| **Charts** | Limited | Built-in (matplotlib, plotly, altair) |
| **Layout** | Automatic | Columns, tabs, sidebar, expanders |
| **Deployment** | Hugging Face Spaces | Streamlit Cloud (free) |

**Rule of thumb:** Use Gradio for quick demos/hackathons. Use Streamlit for dashboards and multi-page apps.

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Build the Dashboard Code</h2>
</div>

We'll write the complete Streamlit app to a Python file. This app has 4 panels:
1. **Upload CSV** — display the data table
2. **AI Insights** — automatic analysis of the dataset
3. **Ask Questions** — chat with your data
4. **Auto Charts** — AI-recommended visualizations

In [ ]:
# ============================================================
# 📝 Write the Streamlit App to a File
# ============================================================

streamlit_code = '''
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import openai
import os
import json

# ---- Page Config ----
st.set_page_config(page_title="AI Data Dashboard", page_icon="📊", layout="wide")

# ---- Header ----
st.title("📊 AI-Powered Data Dashboard")
st.caption("DADS 5250: Generative AI in Practice | Upload CSV → Get AI Insights")

# ---- OpenAI Client ----
client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

def ask_ai(prompt, temperature=0.3):
    """Send a prompt to GPT and return the response."""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

# ---- Sidebar: Upload CSV ----
with st.sidebar:
    st.header("📁 Upload Data")
    uploaded_file = st.file_uploader("Choose a CSV file", type="csv")
    if uploaded_file:
        st.success(f"Loaded: {uploaded_file.name}")

# ---- Main Content ----
if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)

    # Panel 1: Data Table
    tab1, tab2, tab3, tab4 = st.tabs(["📋 Data", "🔍 AI Insights", "💬 Ask Questions", "📈 Auto Charts"])

    with tab1:
        st.subheader("Dataset Overview")
        col1, col2, col3 = st.columns(3)
        col1.metric("Rows", f"{len(df):,}")
        col2.metric("Columns", len(df.columns))
        col3.metric("Missing Values", df.isnull().sum().sum())
        st.dataframe(df.head(50), use_container_width=True)
        with st.expander("Column Details"):
            st.write(df.dtypes.to_frame("Type"))
            st.write(df.describe())

    # Panel 2: AI Insights
    with tab2:
        st.subheader("🔍 AI-Generated Insights")
        if st.button("Generate Insights", key="insights"):
            with st.spinner("Analyzing your data with AI..."):
                summary = f"Columns: {list(df.columns)}\\nShape: {df.shape}\\nSample:\\n{df.head(5).to_string()}\\nStats:\\n{df.describe().to_string()}"
                insights = ask_ai(
                    f"You are a data analyst. Analyze this dataset and provide:\\n"
                    f"1. **Key Findings** (3-5 bullet points)\\n"
                    f"2. **Data Quality Issues** (missing values, outliers, types)\\n"
                    f"3. **Recommended Analysis** (what questions to ask)\\n"
                    f"4. **Business Implications** (actionable insights)\\n\\n"
                    f"Dataset:\\n{summary}"
                )
                st.markdown(insights)

    # Panel 3: Ask Questions
    with tab3:
        st.subheader("💬 Ask Questions About Your Data")
        question = st.text_input("Ask a question about the data:")
        if question:
            with st.spinner("Thinking..."):
                summary = f"Columns: {list(df.columns)}\\nShape: {df.shape}\\nSample:\\n{df.head(5).to_string()}\\nStats:\\n{df.describe().to_string()}"
                answer = ask_ai(
                    f"Based on this dataset, answer the question concisely.\\n\\n"
                    f"Dataset:\\n{summary}\\n\\nQuestion: {question}"
                )
                st.markdown(answer)

    # Panel 4: Auto Charts
    with tab4:
        st.subheader("📈 AI-Recommended Charts")
        if st.button("Generate Charts", key="charts"):
            with st.spinner("AI is choosing the best visualizations..."):
                numeric_cols = df.select_dtypes(include="number").columns.tolist()
                if len(numeric_cols) >= 2:
                    fig, axes = plt.subplots(1, min(3, len(numeric_cols)), figsize=(15, 4))
                    if len(numeric_cols) < 3:
                        axes = [axes] if len(numeric_cols) == 1 else list(axes)
                    for i, col in enumerate(numeric_cols[:3]):
                        axes[i].hist(df[col].dropna(), bins=20, color="#0055d4", alpha=0.7)
                        axes[i].set_title(col, fontsize=12)
                        axes[i].set_facecolor("#1a1a2e")
                    fig.patch.set_facecolor("#0d0d1a")
                    for ax in axes:
                        ax.tick_params(colors="white")
                        ax.title.set_color("white")
                    plt.tight_layout()
                    st.pyplot(fig)

                    # AI interpretation
                    chart_desc = ask_ai(
                        f"I created histograms for these columns: {numeric_cols[:3]}. "
                        f"Stats: {df[numeric_cols[:3]].describe().to_string()}\\n"
                        f"Give a 2-sentence interpretation of what these distributions might tell us."
                    )
                    st.markdown(f"**AI Interpretation:** {chart_desc}")
                else:
                    st.warning("Need at least 2 numeric columns for charts.")

else:
    st.info("👈 Upload a CSV file from the sidebar to get started!")
    st.markdown("""    **Try it with any CSV** — sales data, survey results, financial records, etc.
    
    The AI will automatically:
    - Analyze your data structure
    - Generate business insights
    - Answer your questions
    - Create relevant visualizations
    """)
'''

# Write to file
with open("dashboard_app.py", "w") as f:
    f.write(streamlit_code)

print("✅ Streamlit app written to dashboard_app.py")
print(f"📄 File size: {len(streamlit_code)} characters")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Run in Colab (Tunnel Method)</h2>
</div>

Since Streamlit runs as a web server, we use a **localtunnel** to expose it from Colab. Run the cell below and click the generated URL.

In [ ]:
# ============================================================
# 🚀 Launch Streamlit with Tunnel
# ============================================================
!npm install -g localtunnel 2>/dev/null

import subprocess
import threading
import time

# Start Streamlit in the background
proc = subprocess.Popen(
    ["streamlit", "run", "dashboard_app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)  # Wait for server to start

# Create a tunnel
print("🌐 Starting tunnel...")
!npx localtunnel --port 8501 &
time.sleep(3)
print("\n✅ Click the URL above to open your Streamlit dashboard!")
print("💡 If you see a password page, enter your Colab's external IP (shown above).")
print("\n⚠️ To stop: Kernel → Interrupt or restart the runtime.")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> For a smoother experience, you can run Streamlit locally: <code>pip install streamlit openai pandas matplotlib</code> then <code>streamlit run dashboard_app.py</code>. It opens instantly in your browser with hot-reload. Colab tunnels are just for demo purposes.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Understanding the Code</h2>
</div>

Let's break down the key Streamlit concepts used in our dashboard:

In [ ]:
# ============================================================
# 📖 Streamlit Concepts Cheat Sheet
# ============================================================

concepts = {
    "st.set_page_config()": "Set title, icon, layout (wide/centered) — must be FIRST command",
    "st.title() / st.header()": "Display text headers (like HTML h1/h2)",
    "st.columns(3)": "Create side-by-side columns for layout",
    "st.tabs()": "Create tabbed panels (like browser tabs)",
    "st.sidebar": "Left sidebar for inputs and controls",
    "st.file_uploader()": "Upload files (CSV, images, etc.)",
    "st.dataframe()": "Display an interactive table",
    "st.metric()": "Display a big number with label (like a KPI card)",
    "st.button()": "Clickable button that triggers code",
    "st.text_input()": "Single-line text input",
    "st.spinner()": "Show loading animation while processing",
    "st.pyplot()": "Display a matplotlib figure",
    "st.session_state": "Persist data across reruns (like React state)",
}

print("📖 Streamlit Key Concepts:\n")
for cmd, desc in concepts.items():
    print(f"  {cmd:30s} → {desc}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Streamlit re-runs the entire script top-to-bottom on every interaction. This is why <code>st.session_state</code> is crucial — without it, variables reset on every click. Gradio handles state automatically; Streamlit gives you more control but requires more thought.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What happens every time a user interacts with a Streamlit app (clicks a button, types text)?",
        "options": [
            "Only the changed component re-renders (like React)",
            "The entire Python script re-runs from top to bottom",
            "A WebSocket event is sent to the server for partial update",
            "Nothing happens until the user clicks a 'Submit' button"
        ],
        "answer": 1,
        "explanation": "Streamlit's execution model re-runs the entire script on every interaction. This is simple but means you need st.session_state to persist data between runs. It's the opposite of React's component-based re-rendering."
    },
    {
        "q": "When would you choose Streamlit over Gradio for an AI project?",
        "options": [
            "When you need a quick single-function demo in Colab",
            "When you need a multi-page dashboard with tabs, sidebar, charts, and state management",
            "When you want the simplest possible code (5 lines)",
            "When you need to deploy to Hugging Face Spaces"
        ],
        "answer": 1,
        "explanation": "Streamlit excels at multi-page dashboards with complex layouts (tabs, sidebar, columns), built-in charting, and state management. Gradio is better for quick, single-function demos that run inline in Colab."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add a New Dashboard Panel

Add a 5th tab called **"AI Predictions"** that uses the uploaded data to generate predictions. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Add a Predictions Panel
# ============================================================
# Replace each ----- with the correct value

# This code would go inside the Streamlit app, after tab4
predictions_code = '''
    # Add to the tabs list: ["📋 Data", "🔍 AI Insights", "💬 Ask", "📈 Charts", "🔮 Predictions"]
    with tab5:
        st.subheader("🔮 AI Predictions")
        target_col = st.------(                    # What Streamlit widget for dropdown?
            "Select target column to predict:",
            df.columns.tolist()
        )
        if st.button("Generate Predictions", key="predict"):
            with st.------("AI is analyzing patterns..."):  # What shows a loading animation?
                summary = f"Columns: {list(df.columns)}\\nStats:\\n{df.describe().to_string()}"
                prediction = ask_ai(
                    f"Based on this dataset, predict trends for the column '{target_col}'.\\n"
                    f"Provide: 1) Trend direction, 2) Key drivers, 3) Confidence level.\\n\\n"
                    f"Dataset:\\n{summary}"
                )
                st.------(prediction)               # What displays formatted text?
'''

print("📝 Prediction panel code:")
print(predictions_code)
print("\n💡 Replace the ----- values and add this to your dashboard_app.py")
print("   Hint: selectbox, spinner, markdown")

**📝 Your Observations:** *(double-click to edit this cell)*

1. How does Streamlit's re-run model affect the user experience compared to Gradio? _[Your answer]_

2. What are the advantages of having a sidebar + tabs layout? _[Your answer]_

3. For your hackathon project, would you choose Gradio or Streamlit? Why? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Multi-page app:</b> Use <code>st.page_link()</code> to create a multi-page Streamlit app</li>
    <li><b>Plotly charts:</b> Replace matplotlib with Plotly for interactive charts (<code>st.plotly_chart()</code>)</li>
    <li><b>Deploy to Streamlit Cloud:</b> Push your app to GitHub and deploy at <a href="https://streamlit.io/cloud">streamlit.io/cloud</a> for free</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built an AI-powered data dashboard with Streamlit — CSV upload, insights, Q&A, and charts.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><code style="color: #90caf9;">st.tabs() + st.sidebar</code> — rich dashboard layouts</li>
      <li><code style="color: #90caf9;">st.file_uploader()</code> — easy CSV/file upload</li>
      <li><code style="color: #90caf9;">st.session_state</code> — persist data across reruns</li>
      <li>Streamlit re-runs the <b>entire script</b> on every interaction</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M10: MCP & Guardrails</p>
</div>